# Version status: reviewer visibility workflow

This documented local copy extracts longer waveform windows for visual inspection. Its configured interval is August 25 to September 18, 2023, NOT the entire three-month study. The later September-November run was separate. Confirm the interval and use a new output directory before executing; the setup cell creates directories and later cells download catalog data and write products. The original source code is preserved and outputs are cleared.

Workflow: configure paths and dates; query ComCat; extract timestamp-based windows spanning minute boundaries; filter for visual inspection; save PNGs and a processing audit; manually complete the review table. Missing or incomplete data must be distinguished from inspected non-detections. The 75-second window offers more time for later arrivals but does not guarantee capture of every slow phase or clock offset.


# Reviewer reanalysis: visibility of earthquakes within 100 km

This notebook replaces the original 15-s origin-time window used in
`visibilityofseismicity.ipynb`. For every ComCat event within 100 km of Mount
Rainier, it examines DAS data from **5 s before to 70 s after the catalog origin
time**. This interval includes nominal P, S, and slower surface-wave arrivals at
100 km and provides additional margin for the known DAS clock instability.

The workflow:

1. downloads the ComCat event list for the original study interval;
2. concatenates every one-minute DAS file needed for each 75-s window;
3. saves the extracted waveform and a processing audit table;
4. filters the data from 2--9 Hz and saves one review figure per event; and
5. creates a CSV for recording visual classifications.

Nominal arrival markers are guides only. They are not used to accept or reject
events because the DAS clock was not GPS synchronized.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import glob
import math
import re

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from tqdm.auto import tqdm

from libcomcat.search import search
from libcomcat.dataframes import get_summary_data_frame


# Current raw-data location on sermeq. Files are grouped by month and day.
RAW_ROOT = Path("/1-fnp/petasaur/p-jbod1/rainier/2023")

# New output location. The original products are not overwritten.
OUTPUT_ROOT = Path(
    "/data/data4/veronica-scratch-rainier/swarm_august2023/"
    "quakes_3monthdata/reviewer_100km_origin_minus5_plus70"
)
CUT_DIR = OUTPUT_ROOT / "data"
PLOT_DIR = OUTPUT_ROOT / "plots"
TABLE_DIR = OUTPUT_ROOT / "tables"
for directory in (CUT_DIR, PLOT_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Same catalog interval and search radius used by the original notebook.
CATALOG_START = datetime(2023, 8, 25, 0, 0)
CATALOG_END = datetime(2023, 9, 18, 0, 0)
RAINIER_LAT = 46.879967
RAINIER_LON = -121.726906
SEARCH_RADIUS_KM = 100.0

# Reference point used in the original notebook to represent the DAS array.
DAS_REFERENCE_LAT = 46.785887
DAS_REFERENCE_LON = -121.736573

# Reviewer-corrected analysis window.
SECONDS_BEFORE_ORIGIN = 5.0
SECONDS_AFTER_ORIGIN = 70.0

CHANNEL_MIN = 0
CHANNEL_MAX = 3500  # inclusive
CHANNEL_SPACING_M = 10.0
LOW_CUT_HZ = 2.0
HIGH_CUT_HZ = 9.0
FILTER_ORDER = 2

# Velocities are used only to draw nominal arrival-time guides.
VP_KM_S = 5.5
VS_KM_S = 3.2
SURFACE_VELOCITY_KM_S = 2.0

print("Raw DAS:", RAW_ROOT)
print("Reviewer output:", OUTPUT_ROOT)

## Helper functions

The extraction is based on timestamps stored inside the HDF5 files rather than
assuming that each file contains exactly one complete minute. All minute files
overlapping the requested window are loaded, so events near a minute boundary
can span two or three raw files.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return float(6371.0 * 2.0 * np.arcsin(np.sqrt(a)))


def as_utc_timestamp(value):
    return pd.Timestamp(value).tz_localize("UTC") if pd.Timestamp(value).tzinfo is None else pd.Timestamp(value).tz_convert("UTC")


def minute_sequence(start, end):
    first = as_utc_timestamp(start).floor("min")
    last = as_utc_timestamp(end).floor("min")
    return pd.date_range(first, last, freq="1min")


def find_minute_file(minute):
    stamp = minute.strftime("%Y-%m-%d_%H.%M")
    day_directory = RAW_ROOT / minute.strftime("%m/%d")
    patterns = [
        str(day_directory / f"decimator_{stamp}.00_UTC.h5"),
        str(day_directory / f"decimator_{stamp}.00*"),
        str(day_directory / f"*{stamp}.00*"),
    ]
    matches = []
    for pattern in patterns:
        matches.extend(glob.glob(pattern))
    matches = sorted(set(matches))
    return Path(matches[0]) if matches else None


def read_raw_file(path):
    with h5py.File(path, "r") as handle:
        data = np.asarray(
            handle["Acquisition/Raw[0]/RawData"][:, CHANNEL_MIN : CHANNEL_MAX + 1],
            dtype=np.float32,
        )
        timestamps_us = np.asarray(
            handle["Acquisition/Raw[0]/RawDataTime"], dtype=np.int64
        )
    return data, timestamps_us


def extract_event_window(origin_time):
    origin = as_utc_timestamp(origin_time)
    start = origin - pd.Timedelta(seconds=SECONDS_BEFORE_ORIGIN)
    end = origin + pd.Timedelta(seconds=SECONDS_AFTER_ORIGIN)

    paths = []
    missing_minutes = []
    for minute in minute_sequence(start, end):
        path = find_minute_file(minute)
        if path is None:
            missing_minutes.append(minute.isoformat())
        else:
            paths.append(path)

    if not paths:
        return None, None, paths, missing_minutes

    data_parts, time_parts = [], []
    for path in paths:
        data, timestamps_us = read_raw_file(path)
        data_parts.append(data)
        time_parts.append(timestamps_us)

    data = np.concatenate(data_parts, axis=0)
    timestamps_us = np.concatenate(time_parts)
    order = np.argsort(timestamps_us)
    timestamps_us = timestamps_us[order]
    data = data[order]

    # Remove repeated samples at file boundaries.
    keep = np.r_[True, np.diff(timestamps_us) > 0]
    timestamps_us = timestamps_us[keep]
    data = data[keep]

    start_us = int(round(start.timestamp() * 1e6))
    end_us = int(round(end.timestamp() * 1e6))
    inside = (timestamps_us >= start_us) & (timestamps_us <= end_us)
    return data[inside], timestamps_us[inside], paths, missing_minutes


def event_identifier(row, origin):
    for column in ("id", "event_id", "eventid"):
        if column in row.index and pd.notna(row[column]):
            return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(row[column]))
    return origin.strftime("%Y%m%dT%H%M%S")


def estimate_sampling_rate(timestamps_us):
    if timestamps_us is None or len(timestamps_us) < 2:
        return np.nan
    positive_steps = np.diff(timestamps_us)
    positive_steps = positive_steps[positive_steps > 0]
    return 1e6 / np.median(positive_steps) if len(positive_steps) else np.nan

## Query the original 100-km ComCat catalog

In [ ]:
events = search(
    starttime=CATALOG_START,
    endtime=CATALOG_END,
    latitude=RAINIER_LAT,
    longitude=RAINIER_LON,
    maxradiuskm=SEARCH_RADIUS_KM,
)
events_df = get_summary_data_frame(events).sort_values("time").reset_index(drop=True)
events_df["time"] = pd.to_datetime(events_df["time"], utc=True)
events_df["distance_from_rainier_km"] = [
    haversine_km(RAINIER_LAT, RAINIER_LON, lat, lon)
    for lat, lon in zip(events_df["latitude"], events_df["longitude"])
]
events_df["distance_from_das_reference_km"] = [
    haversine_km(DAS_REFERENCE_LAT, DAS_REFERENCE_LON, lat, lon)
    for lat, lon in zip(events_df["latitude"], events_df["longitude"])
]
events_df.to_csv(TABLE_DIR / "comcat_100km_events.csv", index=False)

print("ComCat events:", len(events_df))
display(events_df.head())

## Extract and save the corrected DAS windows

Each output HDF5 file contains the extracted waveform, timestamps, event
metadata, and the list of raw files used. The summary CSV records missing files
or incomplete windows so that they are not silently interpreted as
non-detections.

In [ ]:
summary_rows = []
expected_duration_s = SECONDS_BEFORE_ORIGIN + SECONDS_AFTER_ORIGIN

for _, row in tqdm(events_df.iterrows(), total=len(events_df), desc="Extracting events"):
    origin = as_utc_timestamp(row["time"])
    event_id = event_identifier(row, origin)
    data, timestamps_us, paths, missing_minutes = extract_event_window(origin)

    result = {
        "event_id": event_id,
        "origin_time_utc": origin.isoformat(),
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "depth_km": row.get("depth", np.nan),
        "magnitude": row.get("magnitude", np.nan),
        "distance_from_rainier_km": row["distance_from_rainier_km"],
        "distance_from_das_reference_km": row["distance_from_das_reference_km"],
        "window_start_relative_origin_s": -SECONDS_BEFORE_ORIGIN,
        "window_end_relative_origin_s": SECONDS_AFTER_ORIGIN,
        "nominal_p_arrival_s": row["distance_from_das_reference_km"] / VP_KM_S,
        "nominal_s_arrival_s": row["distance_from_das_reference_km"] / VS_KM_S,
        "nominal_surface_arrival_s": row["distance_from_das_reference_km"] / SURFACE_VELOCITY_KM_S,
        "raw_file_count": len(paths),
        "raw_files": ";".join(str(path) for path in paths),
        "missing_minutes": ";".join(missing_minutes),
    }

    if data is None or len(data) == 0:
        result.update(
            {
                "status": "no_data",
                "n_samples": 0,
                "sampling_rate_hz": np.nan,
                "actual_duration_s": 0.0,
                "cut_file": "",
            }
        )
        summary_rows.append(result)
        continue

    fs = estimate_sampling_rate(timestamps_us)
    actual_duration_s = (timestamps_us[-1] - timestamps_us[0]) / 1e6
    if missing_minutes:
        status = "partial_missing_raw_file"
    elif actual_duration_s < expected_duration_s - 0.1:
        status = "partial_short_window"
    else:
        status = "complete"

    output_name = f"{origin.strftime('%Y%m%dT%H%M%S')}_{event_id}.h5"
    output_path = CUT_DIR / output_name
    with h5py.File(output_path, "w") as handle:
        handle.create_dataset(
            "Acquisition/Raw[0]/RawData",
            data=data,
            compression="gzip",
        )
        handle.create_dataset(
            "Acquisition/Raw[0]/RawDataTime",
            data=timestamps_us,
            compression="gzip",
        )
        for key, value in result.items():
            handle.attrs[key] = "" if pd.isna(value) else value
        handle.attrs["sampling_rate_hz"] = fs

    result.update(
        {
            "status": status,
            "n_samples": len(timestamps_us),
            "sampling_rate_hz": fs,
            "actual_duration_s": actual_duration_s,
            "cut_file": str(output_path),
        }
    )
    summary_rows.append(result)

processing_summary = pd.DataFrame(summary_rows)
processing_summary.to_csv(TABLE_DIR / "processing_summary.csv", index=False)
print(processing_summary["status"].value_counts(dropna=False))
display(processing_summary.head())

## Filter and plot every available event

The horizontal axis is time relative to the catalog origin. Dashed lines show
nominal P, S, and 2 km/s surface-wave arrival times calculated from the distance
to the DAS reference point. Because the DAS clock may be offset, these markers
are contextual guides rather than matching criteria.

In [ ]:
def filter_das(data, sampling_rate_hz):
    sos = butter(
        FILTER_ORDER,
        [LOW_CUT_HZ, HIGH_CUT_HZ],
        btype="bandpass",
        fs=sampling_rate_hz,
        output="sos",
    )
    return sosfiltfilt(sos, data, axis=0)


def robust_limit(data):
    row_step = max(1, data.shape[0] // 1500)
    col_step = max(1, data.shape[1] // 500)
    sample = np.abs(data[::row_step, ::col_step])
    limit = float(np.nanpercentile(sample, 99.5))
    return limit if np.isfinite(limit) and limit > 0 else 1.0


def plot_event(row):
    if not row["cut_file"]:
        return ""
    with h5py.File(row["cut_file"], "r") as handle:
        data = np.asarray(handle["Acquisition/Raw[0]/RawData"], dtype=np.float32)
        timestamps_us = np.asarray(handle["Acquisition/Raw[0]/RawDataTime"], dtype=np.int64)

    fs = float(row["sampling_rate_hz"])
    filtered = filter_das(data, fs)
    origin_us = int(round(as_utc_timestamp(row["origin_time_utc"]).timestamp() * 1e6))
    relative_s = (timestamps_us - origin_us) / 1e6
    distance_start_km = CHANNEL_MIN * CHANNEL_SPACING_M / 1000.0
    distance_end_km = (CHANNEL_MIN + filtered.shape[1] - 1) * CHANNEL_SPACING_M / 1000.0
    limit = robust_limit(filtered)

    fig, ax = plt.subplots(figsize=(13, 7))
    image = ax.imshow(
        filtered.T,
        cmap="seismic",
        aspect="auto",
        origin="upper",
        vmin=-limit,
        vmax=limit,
        extent=[relative_s[0], relative_s[-1], distance_end_km, distance_start_km],
        interpolation="nearest",
    )
    arrival_guides = [
        (row["nominal_p_arrival_s"], "P at 5.5 km/s", "#111827"),
        (row["nominal_s_arrival_s"], "S at 3.2 km/s", "#0F766E"),
        (row["nominal_surface_arrival_s"], "Surface at 2.0 km/s", "#7C3AED"),
    ]
    ax.axvline(0, color="#B91C1C", linewidth=1.2, label="ComCat origin")
    for arrival_s, label, color in arrival_guides:
        if relative_s[0] <= arrival_s <= relative_s[-1]:
            ax.axvline(arrival_s, color=color, linestyle="--", linewidth=1.1, label=label)

    ax.set_xlabel("Time relative to ComCat origin (s)")
    ax.set_ylabel("Optical distance along fiber (km)")
    ax.set_title(
        f"{row['event_id']} | M {row['magnitude']:.2f} | "
        f"{row['distance_from_das_reference_km']:.1f} km from DAS reference | "
        f"{row['status']}"
    )
    ax.legend(loc="upper right", frameon=True, fontsize=8)
    fig.colorbar(image, ax=ax, label="Filtered DAS amplitude")
    fig.tight_layout()

    plot_path = PLOT_DIR / (Path(row["cut_file"]).stem + ".png")
    fig.savefig(plot_path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return str(plot_path)


plot_paths = []
for _, row in tqdm(processing_summary.iterrows(), total=len(processing_summary), desc="Plotting events"):
    plot_paths.append(plot_event(row))

processing_summary["plot_file"] = plot_paths
processing_summary.to_csv(TABLE_DIR / "processing_summary.csv", index=False)
print("Saved plots:", sum(bool(path) for path in plot_paths))

## Create the visual-review table

Open the PNG files in the `plots` directory and complete the final three
columns. Preserve `uncertain` rather than forcing a binary decision when traffic
or timing instability prevents confident classification.

In [ ]:
review_path = TABLE_DIR / "visual_review.csv"
review_columns = [
    "event_id",
    "origin_time_utc",
    "magnitude",
    "distance_from_rainier_km",
    "distance_from_das_reference_km",
    "status",
    "plot_file",
]
review = processing_summary[review_columns].copy()
review["visual_classification"] = ""  # visible / not_visible / uncertain
review["visible_arrival"] = ""        # P / S / surface / multiple / unclear
review["review_notes"] = ""

if review_path.exists():
    previous = pd.read_csv(review_path)
    preserved = previous[
        ["event_id", "visual_classification", "visible_arrival", "review_notes"]
    ]
    review = review.drop(
        columns=["visual_classification", "visible_arrival", "review_notes"]
    ).merge(preserved, on="event_id", how="left")

review.to_csv(review_path, index=False)
print("Visual-review table:", review_path)
display(review.head())

## Outputs for the reviewer response

- `tables/comcat_100km_events.csv`: queried ComCat population.
- `tables/processing_summary.csv`: extraction audit and nominal arrivals.
- `tables/visual_review.csv`: manual visibility classifications.
- `data/`: 75-s waveform cuts.
- `plots/`: filtered review images.

Report the number of `complete`, `partial`, and `no_data` windows separately.
Only events with usable data should enter the visibility denominator.